In [1]:
import random

# Eight directions: up, down, left, right and the four diagonals
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]


class Courtyard:
    """The environment and its agents: students on a wrapped grid."""

    def __init__(self, n=100, size=30, p=0.3, seed=0):
        self.rng = random.Random(seed)  # each model has its own random generator
        self.n, self.size, self.p, self.tick = n, size, p, 0
        # attribute 1: where each student stands
        self.pos = [(self.rng.randrange(size), self.rng.randrange(size))
                    for _ in range(n)]
        # attribute 2: does each student know the rumour?
        self.knows = [False] * n
        self.knows[0] = True  # student 0 starts the rumour

    def count(self):
        return sum(self.knows)

    def render(self):
        """Draw the courtyard: K = knows the rumour, s = does not, . = empty."""
        rows = [["."] * self.size for _ in range(self.size)]
        for i, (r, c) in enumerate(self.pos):
            if self.knows[i]:
                rows[r][c] = "K"
            elif rows[r][c] == ".":
                rows[r][c] = "s"
        return "\n".join("".join(row) for row in rows)


courtyard = Courtyard(n=12, size=10, seed=1)
print(courtyard.render())
print("Students who know the rumour:", courtyard.count())

......ss..
....s..s..
.........K
..........
...s......
s.........
...s.....s
.......s..
..........
.s........
Students who know the rumour: 1


In [7]:
# ---- add these four methods inside the Courtyard class ----

def move_all(self):
    for i in range(self.n):
        dr, dc = self.rng.choice(DIRS)
        r, c = self.pos[i]
        self.pos[i] = ((r + dr) % self.size, (c + dc) % self.size)  # wrap around

def next_to_knower(self, cell, know_cells):
    r, c = cell
    return any(
        ((r + a) % self.size, (c + b) % self.size) in know_cells
        for a in (-1, 0, 1) for b in (-1, 0, 1)
    )

def spread(self):
    know_cells = {self.pos[i] for i in range(self.n) if self.knows[i]}
    newly = [
        i for i in range(self.n)
        if not self.knows[i]
        and self.next_to_knower(self.pos[i], know_cells)
        and self.rng.random() < self.p
    ]
    for i in newly:
        self.knows[i] = True

def step(self):  # one tick
    self.move_all()
    self.spread()
    self.tick += 1

In [8]:
def run_rumour(model, max_ticks=600, every=10):
    print("tick knowers")
    print(f"{model.tick:4d} {model.count():3d}")

    while model.count() < model.n and model.tick < max_ticks:
        model.step()

        if model.tick % every == 0:
            print(f"{model.tick:4d} {model.count():3d}")

    if model.count() == model.n:
        print(f"Everyone knows after {model.tick} ticks")
    else:
        print(f"Stopped at tick {model.tick} with {model.count()} knowers")


run_rumour(Courtyard(seed=1))

tick knowers
   0   1
  10   4
  20   8
  30  14
  40  26
  50  40
  60  67
  70  91
  80  98
  90 100
Everyone knows after 90 ticks


In [11]:
import statistics

def ticks_until(model, fraction, max_ticks=600):
    target = fraction * model.n
    while model.count() < target and model.tick < max_ticks:
        model.step()
    return model.tick

def average_ticks(p, fraction, seeds=range(30), **kwargs):
    return statistics.mean(
        ticks_until(Courtyard(p=p, seed=s, **kwargs), fraction) for s in seeds
    )

print("Half know after", round(average_ticks(0.3, 0.5), 1), "ticks on average")
print("90 percent know after", round(average_ticks(0.3, 0.9), 1), "ticks on average")

runs = [ticks_until(Courtyard(p=0.3, seed=s), 0.9) for s in range(30)]
print("Single runs to 90 percent: fastest", min(runs), ", slowest", max(runs))

Half know after 43.8 ticks on average
90 percent know after 64.4 ticks on average
Single runs to 90 percent: fastest 50 , slowest 87


In [12]:
for p in [0.1, 0.5, 1.0, 2.0]:
    # TODO 1: half - average ticks until 50 percent know
    half = average_ticks(p, 0.5)

    # TODO 2: ninety - average ticks until 90 percent know
    ninety = average_ticks(p, 0.9)

    # TODO 3: print one line such as p = 0.1: half 81.4, 90% 128.7
    print(f"p = {p:.1f}: half {half:.1f}, 90% {ninety:.1f}")

p = 0.1: half 81.4, 90% 120.7
p = 0.5: half 36.6, 90% 53.0
p = 1.0: half 26.5, 90% 39.6
p = 2.0: half 26.5, 90% 39.6


In [13]:
import statistics

class CourtyardStayHome(Courtyard):
    def __init__(self, stay_home=0.0, **kwargs):
        super().__init__(**kwargs)

        # TODO 1 completed
        k = int(stay_home * self.n)

        # TODO 2 completed
        self.frozen = set(range(self.n - k, self.n))

    def move_all(self):
        # TODO 3 completed
        for i in range(self.n):
            if i not in self.frozen:
                dr, dc = self.rng.choice(DIRS)
                r, c = self.pos[i]
                self.pos[i] = (
                    (r + dr) % self.size,
                    (c + dc) % self.size
                )


for stay_home in [0.0, 0.3, 0.6]:
    half = statistics.mean(
        ticks_until(
            CourtyardStayHome(stay_home=stay_home, seed=s), 0.5
        )
        for s in range(30)
    )

    ninety = statistics.mean(
        ticks_until(
            CourtyardStayHome(stay_home=stay_home, seed=s), 0.9
        )
        for s in range(30)
    )

    everyone = 0

    for s in range(30):
        m = CourtyardStayHome(stay_home=stay_home, seed=s)
        ticks_until(m, 1.0)

        everyone += (m.count() == m.n)

    print(
        f"stay home {stay_home}: half {half:.1f}, "
        f"90% {ninety:.1f}, "
        f"everyone knows in {everyone} of 30 runs"
    )

stay home 0.0: half 43.8, 90% 64.4, everyone knows in 30 of 30 runs
stay home 0.3: half 50.6, 90% 76.4, everyone knows in 30 of 30 runs
stay home 0.6: half 71.8, 90% 112.1, everyone knows in 30 of 30 runs


In [15]:
import random
import statistics

class Town:
    def __init__(self, size=40, empty=0.10, threshold=0.4, seed=0):
        self.rng = random.Random(seed)
        self.size, self.threshold = size, threshold
        cells = size * size
        n_empty = round(empty * cells)
        n_each = (cells - n_empty) // 2
        values = [1] * n_each + [2] * (cells - n_empty - n_each) + [0] * n_empty
        self.rng.shuffle(values)
        self.grid = [values[r * size:(r + 1) * size] for r in range(size)]

    def neighbours(self, r, c):
        s = self.size
        return [(r+a, c+b) for a in (-1, 0, 1)
                for b in (-1, 0, 1) if (a, b) != (0, 0)
                and 0 <= r+a < s and 0 <= c+b < s]

    def agents(self):
        return [(r, c) for r in range(self.size)
                for c in range(self.size) if self.grid[r][c] != 0]

    def empties(self):
        return [(r, c) for r in range(self.size)
                for c in range(self.size) if self.grid[r][c] == 0]

    def average_like_share(self):
        a = self.agents()
        return sum(self.like_share(r, c) for r, c in a) / len(a)

    def discontent(self):
        return [(r, c) for r, c in self.agents()
                if not self.is_content(r, c)]

    def like_share(self, r, c):
        # TODO 1 completed
        occupied = [
            (a, b) for a, b in self.neighbours(r, c)
            if self.grid[a][b] != 0
        ]

        if not occupied:
            return 1.0

        same = sum(
            self.grid[a][b] == self.grid[r][c]
            for a, b in occupied
        )

        return same / len(occupied)

    def is_content(self, r, c):
        # TODO 2 completed
        return self.like_share(r, c) >= self.threshold

    def play_round(self):
        # TODO 3 completed
        movers = self.discontent()
        self.rng.shuffle(movers)

        empties = self.empties()
        moved = 0

        for r, c in movers:
            if not empties:
                break

            j = self.rng.randrange(len(empties))
            er, ec = empties[j]

            self.grid[er][ec] = self.grid[r][c]
            self.grid[r][c] = 0

            empties[j] = (r, c)
            moved += 1

        return moved

In [16]:
town = Town(threshold=0.4, seed=9)

print("Start: like share",
      round(town.average_like_share(), 2),
      ", discontent:", len(town.discontent()))

rounds = 0

while True:
    moved = town.play_round()
    rounds += 1

    if rounds in (1, 2, 5, 10, 15) or moved == 0:
        print(f"Round {rounds}: moved {moved}, "
              f"like share {town.average_like_share():.2f}")

    if moved == 0:
        break

Start: like share 0.5 , discontent: 408
Round 1: moved 408, like share 0.63
Round 2: moved 210, like share 0.71
Round 5: moved 56, like share 0.79
Round 10: moved 12, like share 0.83
Round 15: moved 0, like share 0.83


In [17]:
for threshold in [0.2, 0.4, 0.6]:
    rounds_used, shares = [], []

    for s in range(10):
        town = Town(threshold=threshold, seed=s)
        r = 0

        while r < 100 and town.play_round() > 0:
            r += 1

        rounds_used.append(r)
        shares.append(town.average_like_share())

    print(
        f"Threshold {threshold}: "
        f"rounds {statistics.mean(rounds_used):.1f}, "
        f"like share {statistics.mean(shares):.2f}"
    )

Threshold 0.2: rounds 5.7, like share 0.57
Threshold 0.4: rounds 13.7, like share 0.83
Threshold 0.6: rounds 76.2, like share 0.98


In [18]:
import random
import statistics

SPOTS = ["crate", "door", "shelf", "barrel", "curtain"]
HIDE_PROBS = [0.10, 0.45, 0.05, 0.30, 0.10]


class HidingPlayer:
    def __init__(self, seed):
        self.rng = random.Random(seed)

    def hide(self):
        return self.rng.choices(
            range(len(SPOTS)), weights=HIDE_PROBS
        )[0]


def critic(spot, hidden):
    return 1 if spot == hidden else 0


def play(player, rounds=500, seed=0):
    guard, player = LearningGuard(epsilon, seed), HidingPlayer(1000 + seed)
    wins = 0

    for _ in range(rounds):
        spot = guard.choose()
        reward = critic(spot, player.hide())
        guard.learn(spot, reward)
        wins += reward

    return wins / rounds, guard


class LearningGuard:
    def __init__(self, epsilon, seed):
        self.epsilon = epsilon
        self.rng = random.Random(seed)
        self.checked = [0] * len(SPOTS)
        self.found = [0] * len(SPOTS)

    def estimate(self, i):
        # TODO 1 completed
        return self.found[i] / self.checked[i] if self.checked[i] else 0.0

    def choose(self):
        # TODO 2a completed
        if self.rng.random() < self.epsilon:
            return self.rng.randrange(len(SPOTS))

        # TODO 2b completed
        best = max(self.estimate(i) for i in range(len(SPOTS)))
        return self.rng.choice(
            [i for i in range(len(SPOTS)) if self.estimate(i) == best]
        )

    def learn(self, spot, reward):
        # TODO 3 completed
        self.checked[spot] += 1
        self.found[spot] += reward

In [19]:
def play(epsilon, rounds=500, seed=0):
    guard, player = LearningGuard(epsilon, seed), HidingPlayer(1000 + seed)
    wins = 0

    for _ in range(rounds):
        spot = guard.choose()
        reward = critic(spot, player.hide())
        guard.learn(spot, reward)
        wins += reward

    return wins / rounds, guard

In [20]:
for epsilon in [1.0, 0.6, 0.05, 0.1, 0.3]:
    rates = [play(epsilon, 500, seed)[0] for seed in range(100)]
    print(f"epsilon {epsilon}: success rate {statistics.mean(rates):.3f}")

for epsilon in [0.1, 0.0]:
    rate, guard = play(epsilon, 500, 0)
    print("epsilon", epsilon, "seed 0: success", rate,
          "checks", guard.checked)

epsilon 1.0: success rate 0.201
epsilon 0.6: success rate 0.294
epsilon 0.05: success rate 0.381
epsilon 0.1: success rate 0.394
epsilon 0.3: success rate 0.363
epsilon 0.1 seed 0: success 0.39 checks [14, 410, 8, 56, 12]
epsilon 0.0 seed 0: success 0.318 checks [0, 0, 0, 500, 0]
